In [ ]:
import json, os, sys
assert sys.executable == os.environ['MSSQL_NOTEBOOK_PYTHON']
sys.path.insert(0, os.environ['MSSQL_NOTEBOOK_RUNTIME'])
import mssql_notebook
mssql = mssql_notebook.initialize(os.environ['MSSQL_NOTEBOOK_BROKER_URL'], os.environ['MSSQL_NOTEBOOK_CAPABILITY'], json.loads(os.environ['MSSQL_NOTEBOOK_MESSAGES']))
assert mssql.query('SELECT DB_NAME() AS db').iloc[0, 0] == os.environ['MSSQL_NOTEBOOK_TEST_DATABASE']

In [ ]:
%%tsql --bind df
CREATE TABLE #mixed_notebook (value int, label nvarchar(200));
INSERT INTO #mixed_notebook VALUES (1, N'one'), (2, N'two'), (3, N'three');
SELECT value, label FROM #mixed_notebook ORDER BY value;

In [ ]:
assert list(df.columns) == ['value', 'label']
assert list(df['value']) == [1, 2, 3]
label = "caf\u00e9 \u6771\u4eac '; DROP TABLE #mixed_notebook;--"
query_values = [int(df['value'].sum()) * 7, label]

In [ ]:
%%tsql --bind round_trip --params query_values
INSERT INTO #mixed_notebook (value, label) VALUES (?, ?);
SELECT value, label FROM #mixed_notebook WHERE value = 42;

In [ ]:
assert tuple(round_trip.iloc[0]) == (42, label)
assert mssql.query('SELECT COUNT(*) AS n FROM #mixed_notebook').iloc[0, 0] == 4
retained = round_trip

In [ ]:
%%tsql --all --bind results
PRINT 'mixed notebook';
SELECT COUNT(*) AS n FROM #mixed_notebook;
SELECT SUM(value) AS total FROM #mixed_notebook;

In [ ]:
%%tsql --bind retained
SELECT * FROM #missing_notebook_table;

In [ ]:
assert retained is round_trip
assert [frame.iloc[0, 0] for frame in results] == [4, 48]
mssql.query('BEGIN TRANSACTION')
mssql.query('INSERT INTO #mixed_notebook (value) VALUES (?)', [99])
assert mssql.query('SELECT COUNT(*) AS n FROM #mixed_notebook').iloc[0, 0] == 5

In [ ]:
%%tsql --bind after_rollback
ROLLBACK;
SELECT COUNT(*) AS n FROM #mixed_notebook;

In [ ]:
assert after_rollback.iloc[0, 0] == 4
session = mssql
mssql_notebook.disconnect()
assert get_ipython().find_cell_magic('tsql') is None
assert 'mssql' not in get_ipython().user_ns
try:
    session.query('SELECT 1')
except RuntimeError as error:
    assert str(error) == 'disconnected'
else:
    raise AssertionError('Disconnected notebook still has database access')
print('MSSQL_MIXED_NOTEBOOK_OK')